# Steam Game Recommender — Data Audit & Preparation

Explore Steam reviews and game metadata, check for duplicate interactions, and prepare a smaller dataset for recommender modeling. The initial inspection uses 100K reviews; larger counts are calculated from the full CSV.


In [128]:
import pandas as pd
import numpy as np
import duckdb

## 1. Initial Data Exploration


In [129]:
# Load a sample of interactions for initial inspection
reviews = pd.read_csv(
     "../data/recommendations.csv",
    nrows=100_000
)

games = pd.read_csv("../data/games.csv")

print(reviews.shape)
print(games.shape)

display(reviews.head())
display(games.head())

(100000, 8)
(50872, 13)


,app_id,helpful,funny,date,is_recommended,hours,user_id,review_id
0,975370,0,0,2022-12-12,True,36.3,51580,0
1,304390,4,0,2017-02-17,False,11.5,2586,1
2,1085660,2,0,2019-11-17,True,336.5,253880,2
3,703080,0,0,2022-09-23,True,27.4,259432,3
4,526870,0,0,2021-01-10,True,7.9,23869,4


,app_id,title,date_release,win,mac,linux,rating,positive_ratio,user_reviews,price_final,price_original,discount,steam_deck
0,13500,Prince of Persia: Warrior Within™,2008-11-21,True,False,False,Very Positive,84,2199,9.99,9.99,0.0,True
1,22364,BRINK: Agents of Change,2011-08-03,True,False,False,Positive,85,21,2.99,2.99,0.0,True
2,113020,Monaco: What's Yours Is Mine,2013-04-24,True,True,True,Very Positive,92,3722,14.99,14.99,0.0,True
3,226560,Escape Dead Island,2014-11-18,True,False,False,Mixed,61,873,14.99,14.99,0.0,True
4,249050,Dungeon of the ENDLESS™,2014-10-27,True,True,False,Very Positive,88,8784,11.99,11.99,0.0,True


In [130]:
# Count how many reviews each user has (sample 100k)
reviews_per_user = reviews.groupby('user_id').size()
# Summary statistics
print(reviews_per_user.describe())

# Count users with enough interaction history
for n in [2, 3, 5, 10]:
    count = (reviews_per_user >= n).sum()
    print(f"Users with {n}+ reviews: {count:,}")

count    98803.000000
mean         1.012115
std          0.112861
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          4.000000
dtype: float64
Users with 2+ reviews: 1,160
Users with 3+ reviews: 36
Users with 5+ reviews: 0
Users with 10+ reviews: 0


### User and game review counts (full dataset)

In [131]:
# Count reviews per user across the entire dataset
user_counts = pd.Series(dtype="int64")

#split into chunks for easier computing
for chunk in pd.read_csv(
     "../data/recommendations.csv",
    usecols=["user_id"],
    chunksize=1_000_000
):
    # Count user occurrences within this chunk
    chunk_counts = chunk["user_id"].value_counts()

    # Add counts across chunks
    user_counts = user_counts.add(
        chunk_counts,
        fill_value=0
    )

# Convert accumulated counts back to integers
user_counts = user_counts.astype("int64")

print(user_counts.describe())

for n in [2, 3, 5, 10, 20]:
    count = (user_counts >= n).sum()
    print(f"Users with {n}+ reviews: {count:,}")

count    1.378106e+07
mean     2.986330e+00
std      8.118011e+00
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      3.000000e+00
max      6.045000e+03
dtype: float64
Users with 2+ reviews: 6,208,032
Users with 3+ reviews: 3,771,654
Users with 5+ reviews: 1,910,831
Users with 10+ reviews: 680,812
Users with 20+ reviews: 213,364


In [132]:
# Count positive reviews per user across all chunks
positive_counts = pd.Series(dtype="int64")

for chunk in pd.read_csv(
    "../data/recommendations.csv",
    usecols=["user_id", "is_recommended"],
    chunksize=1_000_000
):
    # Keep only positive reviews
    positive_chunk = chunk[chunk["is_recommended"] == True]

    # Count positive reviews for each user
    chunk_counts = positive_chunk["user_id"].value_counts()

    # Accumulate counts across chunks
    positive_counts = positive_counts.add(
        chunk_counts,
        fill_value=0
    )

positive_counts = positive_counts.astype("int64")

for n in [3, 5, 10, 20]:
    print(f"Users with {n}+ positive reviews: {(positive_counts >= n).sum():,}")

Users with 3+ positive reviews: 3,304,348
Users with 5+ positive reviews: 1,624,066
Users with 10+ positive reviews: 548,339
Users with 20+ positive reviews: 162,033


In [133]:
# Count reviews per game across the complete dataset
game_counts = pd.Series(dtype="int64")

for chunk in pd.read_csv(
    "../data/recommendations.csv",
    usecols=["app_id"],
    chunksize=1_000_000
):
    # Count reviews for each game
    chunk_counts = chunk["app_id"].value_counts()

    # Accumulate counts across chunks
    game_counts = game_counts.add(
        chunk_counts,
        fill_value=0
    )

game_counts = game_counts.astype("int64")

print(game_counts.describe())

for n in [10, 50, 100, 500, 1000]:
    print(f"Games with {n}+ reviews: {(game_counts >= n).sum():,}")

count     37610.000000
mean       1094.251369
std        7689.340463
min           1.000000
25%          13.000000
50%          39.000000
75%         179.750000
max      319492.000000
dtype: float64
Games with 10+ reviews: 31,226
Games with 50+ reviews: 16,977
Games with 100+ reviews: 12,460
Games with 500+ reviews: 5,450
Games with 1000+ reviews: 3,685


### Missing Values

In [134]:
# Check missing values in game metadata
print("Missing values in games:")
print(games.isna().sum())

# Check duplicates in game metadata
print("\nDuplicate game IDs:", games["app_id"].duplicated().sum())

# Check missing values in our review sample
print("\nMissing values in review sample:")
print(reviews.isna().sum())

# Check duplicate user-game pairs in our review sample
print(
    "\nDuplicate user-game pairs:",
    reviews.duplicated(subset=["user_id", "app_id"]).sum()
)

Missing values in games:
app_id            0
title             0
date_release      0
win               0
mac               0
linux             0
rating            0
positive_ratio    0
user_reviews      0
price_final       0
price_original    0
discount          0
steam_deck        0
dtype: int64

Duplicate game IDs: 0

Missing values in review sample:
app_id            0
helpful           0
funny             0
date              0
is_recommended    0
hours             0
user_id           0
review_id         0
dtype: int64

Duplicate user-game pairs: 0


### Playtime Analysis

In [135]:
# Summary statistics for playtime
print(reviews["hours"].describe())

# Compare playtime between positive and negative reviews
print(
    reviews.groupby("is_recommended")["hours"].describe()
)

# Check the proportion of positive reviews
print(
    reviews["is_recommended"].value_counts(normalize=True)
)

count    100000.000000
mean        200.159673
std         236.029457
min           0.000000
25%          30.900000
50%          99.200000
75%         283.200000
max         999.900000
Name: hours, dtype: float64
                  count        mean         std  min   25%    50%     75%  \
is_recommended                                                              
False           15339.0  171.081902  232.870342  0.1  12.6   62.4  230.05   
True            84661.0  205.428025  236.215969  0.0  35.1  105.6  290.80   

                  max  
is_recommended         
False           999.8  
True            999.9  
is_recommended
True     0.84661
False    0.15339
Name: proportion, dtype: float64


### Game Metadata

In [136]:
import json

# Relative path from notebooks/ to data/
metadata_path = "../data/games_metadata.json"

# Read the first line to inspect the JSON structure
with open(metadata_path, "r", encoding="utf-8") as f:
    first_line = f.readline()

# Display the first 1,000 characters
print(first_line[:1000])

{"app_id":13500,"description":"Enter the dark underworld of Prince of Persia Warrior Within, the sword-slashing sequel to the critically acclaimed Prince of Persia: The Sands of Time™. Hunted by Dahaka, an immortal incarnation of Fate seeking divine retribution, the Prince embarks upon a path of both carnage and mystery to defy his preordained death.","tags":["Action","Adventure","Parkour","Third Person","Great Soundtrack","Singleplayer","Platformer","Time Travel","Atmospheric","Classic","Hack and Slash","Time Manipulation","Gore","Fantasy","Story Rich","Dark","Open World","Controller","Dark Fantasy","Puzzle"]}



In [137]:
# Load JSON Lines into a DataFrame
metadata = pd.read_json(
    "../data/games_metadata.json",
    lines=True
)

# Inspect structure
print("Metadata shape:", metadata.shape)
display(metadata.head())

# Check missing descriptions and tags
print("\nMissing values:")
print(metadata[["description", "tags"]].isna().sum())

# Check duplicate game IDs
print("\nDuplicate game IDs:")
print(metadata["app_id"].duplicated().sum())

Metadata shape: (50872, 3)


,app_id,description,tags
0,13500,Enter the dark underworld of Prince of Persia ...,"[Action, Adventure, Parkour, Third Person, Gre..."
1,22364,,[Action]
2,113020,Monaco: What's Yours Is Mine is a single playe...,"[Co-op, Stealth, Indie, Heist, Local Co-Op, St..."
3,226560,Escape Dead Island is a Survival-Mystery adven...,"[Zombies, Adventure, Survival, Action, Third P..."
4,249050,Dungeon of the Endless is a Rogue-Like Dungeon...,"[Roguelike, Strategy, Tower Defense, Pixel Gra..."



Missing values:
description    0
tags           0
dtype: int64

Duplicate game IDs:
0


In [138]:
# Check descriptions that are empty strings
empty_descriptions = (
    metadata["description"].str.strip() == ""
).sum()

# Check games with empty tag lists
empty_tags = metadata["tags"].apply(
    lambda tags: len(tags) == 0
).sum()

# Check matching game IDs
matched_games = metadata["app_id"].isin(games["app_id"]).sum()

print("Empty descriptions:", empty_descriptions)
print("Empty tags:", empty_tags)
print("Matching game IDs:", matched_games)
print("Total metadata games:", len(metadata))

Empty descriptions: 10373
Empty tags: 1244
Matching game IDs: 50872
Total metadata games: 50872


## 2. Data Cleaning & Deduplication


In [139]:
# Check duplicate user-game pairs and metadata coverage
seen_pairs = set()
duplicate_pairs = 0
reviewed_game_ids = set()

for chunk in pd.read_csv(
    "../data/recommendations.csv",
    usecols=["user_id", "app_id"],
    chunksize=1_000_000
):
    # Count duplicate user-game pairs within this chunk
    duplicate_pairs += chunk.duplicated(
        subset=["user_id", "app_id"]
    ).sum()

    # Collect game IDs for metadata matching
    reviewed_game_ids.update(chunk["app_id"].unique())

print("Duplicate pairs within chunks:", duplicate_pairs)

# Compare reviewed games against metadata
matched = len(reviewed_game_ids.intersection(set(metadata["app_id"])))

print("Reviewed games:", len(reviewed_game_ids))
print("Reviewed games with metadata:", matched)
print("Metadata coverage:", round(matched / len(reviewed_game_ids) * 100, 2), "%")

Duplicate pairs within chunks: 8
Reviewed games: 37610
Reviewed games with metadata: 37610
Metadata coverage: 100.0 %


### Duplicate user-game reviews

In [140]:
# Connect to an in-memory DuckDB database
con = duckdb.connect()

# Create a temporary table of duplicate user-game pairs
con.execute("""
    CREATE OR REPLACE TEMP TABLE duplicate_pairs AS
    SELECT
        user_id,
        app_id,
        COUNT(*) AS review_count
    FROM read_csv_auto('../data/recommendations.csv')
    GROUP BY user_id, app_id
    HAVING COUNT(*) > 1
""")

# Count duplicate pairs and extra review rows
duplicate_summary = con.execute("""
    SELECT
        COUNT(*) AS duplicate_pairs,
        SUM(review_count - 1) AS extra_review_rows
    FROM duplicate_pairs
""").df()

# Inspect duplicate pairs
duplicate_details = con.execute("""
    SELECT *
    FROM duplicate_pairs
    ORDER BY review_count DESC
""").df()

display(duplicate_summary)
display(duplicate_details)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,duplicate_pairs,extra_review_rows
0,21,21.0


,user_id,app_id,review_count
0,14227998,570940,2
1,10858200,945360,2
2,3081752,617290,2
3,5485206,220,2
4,10442146,1689910,2
5,11640276,1142710,2
6,3126060,225540,2
7,11571854,1145360,2
8,5969373,231140,2
9,2433696,440,2


### Check dates and recommendation changes

In [141]:
# Inspect the original reviews belonging to duplicated user-game pairs
duplicate_reviews = con.execute("""
    SELECT
        r.user_id,
        r.app_id,
        r.review_id,
        r.date,
        r.is_recommended,
        r.hours
    FROM read_csv_auto('../data/recommendations.csv') AS r
    
    INNER JOIN duplicate_pairs AS d
        ON r.user_id = d.user_id
        AND r.app_id = d.app_id
    ORDER BY r.user_id, r.app_id, r.date DESC
""").df()
    #show newest review first date DESC
    
#Result = each user-game, dates, pos/neg review, hours played
display(duplicate_reviews)

,user_id,app_id,review_id,date,is_recommended,hours
0,1085990,1205960,36533636,2021-12-04,False,3.0
1,1085990,1205960,38113748,2021-12-03,True,3.0
2,2433696,440,24887439,2021-12-04,True,658.0
3,2433696,440,24170089,2021-12-02,True,658.0
4,3054178,1454400,15321463,2021-12-07,True,21.4
5,3054178,1454400,23765890,2021-12-03,True,21.4
6,3070422,1118200,25753093,2022-06-12,True,53.2
7,3070422,1118200,26644717,2022-06-07,True,53.2
8,3081752,617290,6259392,2019-10-11,True,55.3
9,3081752,617290,20332724,2019-08-27,True,55.3


In [142]:
# Find user-game pairs where is_recommended changed
changed_reviews = duplicate_reviews.groupby(
    ['user_id', 'app_id']
).filter(
    lambda x : x['is_recommended'].nunique() > 1
)
display(changed_reviews)

,user_id,app_id,review_id,date,is_recommended,hours
0,1085990,1205960,36533636,2021-12-04,False,3.0
1,1085990,1205960,38113748,2021-12-03,True,3.0
18,5017461,282440,35704524,2016-01-15,True,10.8
19,5017461,282440,35936315,2014-09-26,False,10.8
20,5235993,266840,13388299,2016-04-22,True,761.3
21,5235993,266840,14254084,2015-06-11,False,761.3
26,5969373,231140,38338009,2015-08-17,True,3.6
27,5969373,231140,37305025,2015-03-01,False,3.6
28,8354604,270880,737242,2022-01-06,True,143.0
29,8354604,270880,4894532,2021-12-16,False,143.0


### Keep the most recent review per user-game pair

For users who reviewed the same game more than once, keep their latest review. `review_id` is used to break ties when review dates match.

In [143]:
# Keep the latest review per user-game pair
con.execute("""
    CREATE OR REPLACE VIEW reviews_deduped AS
    -- Drop the temporary row number
    SELECT * EXCLUDE (rn)
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY user_id, app_id
                    -- Break ties using review_id
                ORDER BY date DESC, review_id DESC
            ) AS rn
        FROM read_csv_auto('../data/recommendations.csv')
    )
    WHERE rn = 1
""")

In [144]:
#2) Check whether any duplicate user-game pairs remain
con.execute("""
    SELECT
        user_id,
        app_id,
        COUNT(*) AS review_count
    FROM reviews_deduped
    GROUP BY user_id, app_id
    HAVING COUNT(*) > 1
    LIMIT 10
""").df()
#RESULT = NONE bc no duplicate user-game pairs

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,user_id,app_id,review_count


## 3. Modeling Dataset Preparation


### A1 — Identify Eligible Users

Start with users who have at least five positive reviews, so there are enough liked games to hold some out during evaluation. This focuses our initial models on active reviewers.

In [145]:
#A1) Identify users with at least 5 positive reviews
eligible_users = positive_counts[positive_counts>=5].index
print(f"Eligible users: {len(eligible_users):,}")
eligible_users

Eligible users: 1,624,066


Index([       0,       13,       14,       36,       45,       58,       76,
             85,       89,       95,
       ...
       14305954, 14305966, 14305973, 14305989, 14305996, 14306001, 14306002,
       14306011, 14306019, 14306020],
      dtype='int64', name='user_id', length=1624066)

### A2 — Sample Users and Extract Reviews

Randomly sample 50,000 eligible users and keep all their reviews (positive and negative).

In [146]:
#A2) Randomly select 50,000 eligible users (5+ positive reviews)
sampled_users = eligible_users.to_series().sample(
    n=50000,
    random_state=42
).index

print(f'Sampled user: {len(sampled_users):,}')
display(sampled_users)

Sampled user: 50,000


Index([ 9605747,  1137077,  8728935,  7316840,  7562127,  7014863, 10453602,
        3973049,  2308224, 10128267,
       ...
       12402797, 10677656, 12940949, 13478921, 12248761, 10309378, 10075398,
        4950434, 12852689,  3384127],
      dtype='int64', name='user_id', length=50000)

In [147]:
# Convert sampled user IDs to a DataFrame
sampled_users_df = sampled_users.to_frame(index=False)

# Register the selected users with DuckDB
con.register("sampled_users", sampled_users_df)

# Retrieve all reviews for selected users
sampled_reviews = con.execute("""
    SELECT r.*      -- (only * from r)
    FROM reviews_deduped as r
    INNER JOIN sampled_users AS s
        ON r.user_id = s.user_id
""").df()
display(sampled_reviews.shape, sampled_reviews.head())
print(f"Sampled interactions: {len(sampled_reviews):,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(642760, 8)

,app_id,helpful,funny,date,is_recommended,hours,user_id,review_id
0,504230,0,0,2021-12-30,True,0.2,1651900,21076513
1,635260,0,0,2021-03-07,True,132.4,1674641,2927402
2,477160,0,0,2022-08-01,True,9.1,1688509,21726859
3,440,0,0,2019-04-03,True,46.2,1692836,23978312
4,282070,5,0,2019-02-06,True,89.0,1708401,38830767


Sampled interactions: 642,760


### A3 — Verify Sampled Dataset


In [148]:
#A3) Check how many unique users and unique games
print("Unique users:", sampled_reviews['user_id'].nunique())
print("Unique games:", sampled_reviews["app_id"].nunique())
print("Total interactions:", len(sampled_reviews))

Unique users: 50000
Unique games: 22521
Total interactions: 642760


In [149]:
# Verify minimum positive interactions per sampled user
    #for each user, find count POSITIVE reviews (use sum)
positive_per_user = (sampled_reviews
                     .groupby('user_id')['is_recommended'].sum()
)
display(positive_per_user)
print("Minimum positive reviews:", positive_per_user.min())
print("Users with fewer than 5:", (positive_per_user < 5).sum())
print("Users with no positive reviews:",
      sampled_reviews["user_id"].nunique() - len(positive_per_user))

user_id
232         18
246          6
1281         5
1951         5
2159        13
            ..
14304678     5
14304993    10
14305121     8
14305281     6
14305611     6
Name: is_recommended, Length: 50000, dtype: int64

Minimum positive reviews: 5
Users with fewer than 5: 0
Users with no positive reviews: 0


### A4 — Calculate Density and Sparsity


In [150]:
#A4) density and sparsity

# Number of unique users and games
n_users = sampled_reviews["user_id"].nunique()
n_games = sampled_reviews["app_id"].nunique()

# Total observed interactions
n_interactions = len(sampled_reviews)

# Calculate interaction matrix density and sparsity
# Observed interactions / possible interactions
density = n_interactions / (n_users * n_games)

sparsity = 1 - density

print(f"Density: {density:.4%}")
print(f"Sparsity: {sparsity:.4%}")

Density: 0.0571%
Sparsity: 99.9429%


Adding more users may provide more training interactions, but it does not necessarily make the user-game matrix denser.

### A5 — Analyze Game Interaction Distribution


In [151]:
#A5) Count how many reviews each game received
sample_game_counts = sampled_reviews["app_id"].value_counts()

# Summary statistics
display(sample_game_counts.describe())

# Count games with at least a certain number of reviews
for n in [1, 5, 10, 20, 50, 100]:
    count = (sample_game_counts >= n).sum()
    print(f"Games with {n}+ reviews: {count:,}")

count    22521.000000
mean        28.540473
std        126.679064
min          1.000000
25%          1.000000
50%          3.000000
75%         10.000000
max       3813.000000
Name: count, dtype: float64

Games with 1+ reviews: 22,521
Games with 5+ reviews: 9,047
Games with 10+ reviews: 5,910
Games with 20+ reviews: 3,826
Games with 50+ reviews: 2,027
Games with 100+ reviews: 1,181


The 50K-user sample includes 22,521 games, but the median game has only 3 reviews. We will keep the full catalog for now and see how the models handle less-popular games.

### A6 — Analyze Games with 10+ Reviews


In [152]:
#A6) Identify games with at least 10 reviews
popular_games = sample_game_counts[sample_game_counts >= 10].index

# Check how many interactions/reviews belong to those games
popular_interactions = sampled_reviews[
    sampled_reviews["app_id"].isin(popular_games)
]

print(f"Interactions in games with 10+ reviews: {len(popular_interactions):,}")
print(f"Percentage of all interactions: {len(popular_interactions) / len(sampled_reviews):.2%}")

Interactions in games with 10+ reviews: 597,726
Percentage of all interactions: 92.99%


Games with at least 10 reviews make up 26.2% of the catalog but account for 92.99% of the interactions. The remaining games form a long tail, which could make recommendations harder for less-popular titles.

### A7 — Analyze Positive vs. Negative Reviews

In [153]:
# A7) Analyze Positive vs. Negative Reviews

# Count positive and negative reviews
review_counts = sampled_reviews['is_recommended'].value_counts()

# Calculate the proportion of each review type
review_proportions = sampled_reviews["is_recommended"].value_counts(
    normalize=True
)

print("Review counts:")
print(review_counts)

print("\nReview proportions:")
print(review_proportions)

Review counts:
is_recommended
True     559708
False     83052
Name: count, dtype: int64

Review proportions:
is_recommended
True     0.870788
False    0.129212
Name: proportion, dtype: float64


For the first models, positive reviews will indicate games a user liked. We will keep negative reviews as well, including for filtering previously reviewed games. Playtime could be another signal to test later, but more hours do not always mean a user liked a game.

### A8 — Export Modeling Dataset

In [154]:
from pathlib import Path

# Create a directory for processed data
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Save all sampled reviews, including positive and negative feedback
sampled_reviews.to_parquet(
    output_dir / "sampled_reviews.parquet",
    index=False
)

print("Saved:", output_dir / "sampled_reviews.parquet")
print("Dataset shape:", sampled_reviews.shape)

Saved: ../data/processed/sampled_reviews.parquet
Dataset shape: (642760, 8)


## Summary

The final sample contains 50,000 users, 22,521 games, and 642,760 reviews after deduplication. The user-game matrix is highly sparse, and most reviews are concentrated among a relatively small set of games. About 87% of reviews are positive.

Next, we will split user histories into training and test interactions, build a popularity baseline, and compare recommendation models using ranking metrics.
